# Import packages and data loading

In [141]:
import time
import sys
import json
import pandas as pd
import random

import os
from pathlib import Path

import tempfile

from typing import Any, Dict, List, Tuple, Iterable
import pyomo.environ as pyo

project_root = Path.cwd().resolve().parent
sys.path.insert(0, str(project_root))

# Phase 4 - Data Generation

In [142]:
def generate_instance(n, m, score_range=(0, 100), quota_factor=2, complete=True, strict=False):
    """
    n: number of applicants
    m: number of universities
    score_range: range of scores
    quota_factor: factor for determining capacity
    complete: if True, each applicant applies to all universities (complete list)
    strict: if True, for each college all applicant scores are unique (no ties), by adding a tiny epsilon. 
    """
    # generating quotas for each university
    quotas = {}
    max_quota = int(max(2, n // m) * quota_factor)
    for j in range(m):
        quotas[j] = random.randint(1, max_quota)
    
    # generating preferences and scores
    preferences = {}
    scores = {}
    for i in range(n):
        # list of universities
        if complete:
            colleges = list(range(m))
        else:
            k = max(1, int(m * 0.6))
            colleges = random.sample(range(m), k)
        random.shuffle(colleges)
        preferences[i] = colleges  # order of preference
        
        # generating scores for each university in the list
        for j in colleges:
            base_score = random.randint(score_range[0], score_range[1])
            if strict:
                score = base_score * (n + 1) + i
            else:
                score = base_score
            scores[f"{i},{j}"] = score

    return {
        "n": n,
        "m": m,
        "preferences": preferences,
        "scores": scores,
        "quotas": quotas
    }

def save_instance(data, filename):
    with open(filename, 'w') as f:
        json.dump(data, f, indent=2)

In [143]:
random.seed(42)

output_dir = project_root / "data" / "generated"
output_dir.mkdir(parents=True, exist_ok=True)

small_filename = "instance_small.json"
strict_small_filename = "instance_strict_small.json"
medium_filename = "instance_medium.json"
strict_medium_filename = "instance_strict_medium.json"
large_filename = "instance_large.json"
strict_large_filename = "instance_strict_large.json"

small_n, small_m = 10, 5
medium_n, medium_m = 50, 20
large_n, large_m = 1000, 20

small_score_min, small_score_max = 0, 20
medium_score_min, medium_score_max = 0, 50
large_score_min, large_score_max = 0, 50

small_quota_factor, medium_quota_factor, large_quota_factor = 1, 2, 2

In [144]:
# # ---- Generate small instance ----
# small_data = generate_instance(
#     n=small_n,
#     m=small_m,
#     score_range=(small_score_min, small_score_max),
#     quota_factor=small_quota_factor,
#     complete=True,
#     strict=False
# )
# small_path = output_dir / small_filename
# save_instance(small_data, str(small_path))
# print(f"Saved small instance to: {small_path}")

# # # ---- Generate medium instance ----
# # medium_data = generate_instance(
# #     n=medium_n,
# #     m=medium_m,
# #     score_range=(medium_score_min, medium_score_max),
# #     quota_factor=medium_quota_factor,
# #     complete=True,
# #     strict=False
# # )
# # medium_path = output_dir / medium_filename
# # save_instance(medium_data, str(medium_path))
# # print(f"Saved medium instance to: {medium_path}")

# # # ---- Generate large instance ----
# # large_data = generate_instance(
# #     n=large_n,
# #     m=large_m,
# #     score_range=(large_score_min, large_score_max),
# #     quota_factor=large_quota_factor,
# #     complete=True,
# #     strict=False
# # )
# # large_path = output_dir / large_filename
# # save_instance(large_data, str(large_path))
# # print(f"Saved large instance to: {large_path}")

# # # ---- Generate strict small instance ----
# # strict_small_data = generate_instance(
# #     n=small_n,
# #     m=small_m,
# #     score_range=(small_score_min, small_score_max),
# #     quota_factor=small_quota_factor,
# #     complete=True,
# #     strict=True
# # )
# # strict_small_path = output_dir / strict_small_filename
# # save_instance(strict_small_data, str(strict_small_path))
# # print(f"Saved strict small instance to: {strict_small_path}")

# # # ---- Generate strict medium instance ----
# # medium_strict_data = generate_instance(
# #     n=medium_n,
# #     m=medium_m,
# #     score_range=(medium_score_min, medium_score_max),
# #     quota_factor=medium_quota_factor,
# #     complete=True,
# #     strict=True
# # )
# # strict_medium_path = output_dir / strict_medium_filename
# # save_instance(medium_strict_data, str(strict_medium_path))
# # print(f"Saved strict medium instance to: {strict_medium_path}")

# # # ---- Generate strict large instance ----
# # large_strict_data = generate_instance(
# #     n=large_n,
# #     m=large_m,
# #     score_range=(large_score_min, large_score_max),
# #     quota_factor=large_quota_factor,
# #     complete=True,
# #     strict=True
# # )

# # strict_large_path = output_dir / strict_large_filename
# # save_instance(large_strict_data, str(strict_large_path))
# # print(f"Saved strict large instance to: {strict_large_path}")


## Loading Datasets

In [145]:
def load_instance(filename: str | Path | None = None, data: Dict[str, Any] | None = None):
    """Load a college admission instance from a JSON file or a preloaded dict."""
    if data is not None:
        instance = data
    else:
        with open(filename, "r", encoding="utf-8") as handle:
            instance = json.load(handle)

    if "preferences" in instance and isinstance(instance["preferences"], dict):
        instance["preferences"] = {int(k): [int(v2) for v2 in v] for k, v in instance["preferences"].items()}
    if "quotas" in instance and isinstance(instance["quotas"], dict):
        instance["quotas"] = {int(k): int(v) for k, v in instance["quotas"].items()}

    if "scores" in instance and isinstance(instance["scores"], dict):
        normalized_scores = {}
        for key, value in instance["scores"].items():
            if isinstance(key, tuple):
                normalized_scores[(int(key[0]), int(key[1]))] = int(value)
            else:
                i_str, j_str = key.split(",")
                normalized_scores[(int(i_str), int(j_str))] = int(value)
        instance["scores"] = normalized_scores

    instance["n"] = int(instance["n"])
    instance["m"] = int(instance["m"])
    return instance

def _normalize_data(data: Dict[str, Any]) -> Dict[str, Any]:
    if isinstance(data, (str, Path)):
        return load_instance(data)
    return load_instance(data=data)

In [146]:
dataset_small = json.load(open(r'../data/generated/instance_small.json', 'r', encoding='utf-8'))
print("Loaded small instance:\n   n={}, m={}".format(dataset_small['n'], dataset_small['m']))

dataset_medium = json.load(open(r'../data/generated/instance_medium.json', 'r', encoding='utf-8'))
print("Loaded medium instance:\n   n={}, m={}".format(dataset_medium['n'], dataset_medium['m']))

dataset_strict_small = json.load(open(r'../data/generated/instance_strict_small.json', 'r', encoding='utf-8'))
print("Loaded strictly-ranked small instance:\n   n={}, m={}".format(dataset_strict_small['n'], dataset_strict_small['m']))

dataset_strict_medium = json.load(open(r'../data/generated/instance_strict_medium.json', 'r', encoding='utf-8'))
print("Loaded strictly-ranked medium instance:\n   n={}, m={}".format(dataset_strict_medium['n'], dataset_strict_medium['m']))

Loaded small instance:
   n=10, m=5
Loaded medium instance:
   n=50, m=20
Loaded strictly-ranked small instance:
   n=10, m=5
Loaded strictly-ranked medium instance:
   n=50, m=20


In [147]:
data_small = _normalize_data(dataset_small)
data_medium = _normalize_data(dataset_medium)
data_strict_small = _normalize_data(dataset_strict_small)
data_strict_medium = _normalize_data(dataset_strict_medium)

del dataset_small, dataset_medium, dataset_strict_small, dataset_strict_medium

# Phase 5 - Models

## Basic Functions for Building Models

In [148]:
def _applications(data: Dict[str, Any]) -> List[Tuple[int, int]]:
    preferences = data["preferences"]
    n = data["n"]
    applications: List[Tuple[int, int]] = []
    for i in range(n):
        for j in preferences.get(i, []):
            applications.append((i, j))
    return applications

def _score_lists(data: Dict[str, Any]) -> Dict[int, List[int]]:
    scores = data["scores"]
    m = data["m"]
    score_sets: Dict[int, set[int]] = {j: set() for j in range(m)}
    for (i, j), value in scores.items():
        score_sets[j].add(value)
    return {j: sorted(score_sets[j]) for j in range(m)}

def _build_common_components(model: pyo.ConcreteModel, data: Dict[str, Any]) -> None:
    n = data["n"]
    m = data["m"]
    preferences = data["preferences"]
    scores = data["scores"]
    quotas = data["quotas"]

    model.A = pyo.Set(initialize=range(n))
    model.C = pyo.Set(initialize=range(m))
    model.E = pyo.Set(initialize=_applications(data), dimen=2)

    def rank_init(model, i, j):
        return preferences[i].index(j)

    def score_init(model, i, j):
        return scores[(i, j)]

    def quota_init(model, j):
        return quotas.get(j, 0)

    model.u = pyo.Param(model.C, initialize=quota_init, within=pyo.NonNegativeIntegers)
    model.r = pyo.Param(model.E, initialize=rank_init, within=pyo.NonNegativeIntegers)
    model.s = pyo.Param(model.E, initialize=score_init, within=pyo.NonNegativeIntegers)
    model.x = pyo.Var(model.E, within=pyo.Binary)

    def one_per_student(model, i):
        return sum(model.x[i, j] for (i2, j) in model.E if i2 == i) <= 1

    def capacity(model, j):
        return sum(model.x[i, j2] for (i, j2) in model.E if j2 == j) <= model.u[j]

    model.OnePerStudent = pyo.Constraint(model.A, rule=one_per_student)
    model.Capacity = pyo.Constraint(model.C, rule=capacity)

In [149]:
def print_model_summary(model: pyo.ConcreteModel, print_assignments: bool = True) -> None:
    """
    Print a clear summary of a solved Pyomo model:
    - objective value
    - number of assigned students
    - average preference rank
    - college loads
    - cutoff scores (minimum score of admitted students per college)
    - list of assignments
    - list of unassigned students
    """
    first_var = next(iter(model.component_objects(pyo.Var, active=True)))
    first_idx = next(iter(first_var))
    _ = pyo.value(first_var[first_idx])

    obj_val = pyo.value(model.Objective) if hasattr(model, 'Objective') else None

    # Extract assignment
    assignment = {}
    total_rank = 0
    for i in model.A:
        assigned = None
        for j in model.C:
            if (i, j) in model.E and pyo.value(model.x[i, j]) > 0.5:
                assigned = j
                total_rank += pyo.value(model.r[i, j])
                break
        assignment[i] = assigned

    assigned_students = [i for i, j in assignment.items() if j is not None]
    num_assigned = len(assigned_students)

    # College loads
    college_loads = {j: 0 for j in model.C}
    for i, j in assignment.items():
        if j is not None:
            college_loads[j] += 1

    # Cutoff scores
    cutoffs = {}
    for j in model.C:
        scores_j = []
        for i in model.A:
            if (i, j) in model.E and pyo.value(model.x[i, j]) > 0.5:
                scores_j.append(pyo.value(model.s[i, j]))
        cutoffs[j] = min(scores_j) if scores_j else 0

    # Print summary
    print("\n" + "="*60)
    print("MODEL SUMMARY")
    print("="*60)
    if obj_val is not None:
        print(f"Objective value: {obj_val:.2f}")
    else:
        print("Objective: not available")
    print(f"Students assigned: {num_assigned} out of {len(model.A)}")
    if num_assigned > 0:
        avg_rank = total_rank / num_assigned
        print(f"Average rank: {avg_rank:.2f}")
        print(f"Total rank sum: {total_rank}")
    else:
        print("No students assigned.")
    print("\nCollege loads:", college_loads)
    print("Cutoff scores:", cutoffs)
    if print_assignments:
        print("\nAssignments:")
        for i, j in assignment.items():
            if j is not None:
                print(f"  Student {i} to College {j} (rank {pyo.value(model.r[i, j])})")
    unassigned = [i for i, j in assignment.items() if j is None]
    if unassigned:
        print("Unassigned students:", unassigned)
    print("="*60 + "\n")

In [150]:
solver = pyo.SolverFactory("cplex")

In [151]:
def build_model(data: Dict[str, Any] | str | Path, formulation: str = "SO-BB") -> pyo.ConcreteModel:
    """
    Build one formulation as a Pyomo model.
    """
    data = _normalize_data(data)
    model = pyo.ConcreteModel()
    _build_common_components(model, data)

    if formulation == "SO-BB":
        return _build_so_bb(model, data)
    elif formulation == "SO-NW-CUT":
        return _build_so_nw_cut(model, data)
    elif formulation == "MIN-CUT":
        return _build_min_cut(model, data)
    elif formulation == "MSMR-CUT":
        return _build_msmr_cut(model, data)
    elif formulation == "SO-NW-BIN-CUT":
        return _build_so_nw_bin_cut(model, data)
    elif formulation == "MIN-BIN-CUT":
        return _build_min_bin_cut(model, data)
    elif formulation == "MSMR-BIN-CUT":
        return _build_msmr_bin_cut(model, data)
    elif formulation == "MSMR-EF":
        return _build_msmr_ef(model, data)
    elif formulation == "SO-H-NW-CUT":
        return _build_so_h_nw_cut(model, data)
    elif formulation == "SO-H-NW-BIN-CUT":
        return _build_so_h_nw_bin_cut(model, data)
    elif formulation == "SO-C-NW-CUT":
        return _build_so_c_nw_cut(model, data)
    elif formulation == "SO-C-NW-BIN-CUT":
        return _build_so_c_nw_bin_cut(model, data)

## Section 2 - The Gale-Shapley Model

### Baiou-Balinski (SO-BB) Formulation

In [152]:
def _build_so_bb(model: pyo.ConcreteModel, data: Dict[str, Any]) -> pyo.ConcreteModel:
    """
    SO-BB: Student-Optimal Baïou-Balinski formulation.
    """
    # --- Baïou-Balinski stability constraints
    def baiou_balinski_rule(model, i, j):
        rank_ij = model.r[i, j]
        preferred_or_equal = sum(
            model.x[i, h]
            for (ii, h) in model.E
            if ii == i and model.r[ii, h] <= rank_ij
        )
        higher_score = sum(
            model.x[h, j]
            for (h, j2) in model.E
            if j2 == j and model.s[h, j] > model.s[i, j]
        )
        return preferred_or_equal * model.u[j] + higher_score >= model.u[j]

    model.StabilityConstraint = pyo.Constraint(model.E, rule=baiou_balinski_rule)

    # --- Objective
    def objective(model):
        return sum(model.r[i, j] * model.x[i, j] for (i, j) in model.E)

    model.Objective = pyo.Objective(rule=objective, sense=pyo.minimize)
    return model

In [153]:
so_bb_small = build_model(data_strict_small, formulation="SO-BB")
_ = solver.solve(so_bb_small)
print_model_summary(so_bb_small)


MODEL SUMMARY
Objective value: 7.00
Students assigned: 8 out of 10
Average rank: 0.88
Total rank sum: 7

College loads: {0: 2, 1: 2, 2: 1, 3: 2, 4: 1}
Cutoff scores: {0: 166, 1: 132, 2: 136, 3: 161, 4: 138}

Assignments:
  Student 0 to College 1 (rank 0)
  Student 1 to College 0 (rank 0)
  Student 4 to College 2 (rank 1)
  Student 5 to College 1 (rank 0)
  Student 6 to College 4 (rank 1)
  Student 7 to College 3 (rank 0)
  Student 8 to College 3 (rank 2)
  Student 9 to College 0 (rank 3)
Unassigned students: [2, 3]



In [154]:
so_bb_medium = build_model(data_medium, formulation="SO-BB")
_ = solver.solve(so_bb_medium)
print_model_summary(so_bb_medium, print_assignments=False)


MODEL SUMMARY
Objective value: 34.00
Students assigned: 50 out of 50
Average rank: 0.68
Total rank sum: 34

College loads: {0: 3, 1: 2, 2: 4, 3: 4, 4: 3, 5: 2, 6: 2, 7: 2, 8: 2, 9: 3, 10: 4, 11: 4, 12: 3, 13: 2, 14: 2, 15: 3, 16: 1, 17: 1, 18: 1, 19: 2}
Cutoff scores: {0: 26, 1: 31, 2: 1, 3: 6, 4: 6, 5: 27, 6: 7, 7: 30, 8: 35, 9: 31, 10: 1, 11: 8, 12: 40, 13: 9, 14: 34, 15: 9, 16: 48, 17: 7, 18: 29, 19: 33}



### SO-NW-Cut Formulation

In [155]:
def _build_so_nw_cut(model: pyo.ConcreteModel, data: Dict[str, Any]) -> pyo.ConcreteModel:
    # --- Cutoff variables and parameters
    scores = data["scores"]
    big_m = max(scores.values()) + 2
    epsilon = 1e-6

    model.t = pyo.Var(model.C, within=pyo.NonNegativeReals, bounds=(0, big_m))
    model.f = pyo.Var(model.C, within=pyo.Binary)

    # --- Cutoff constraints (5) and (6)
    def cutoff_upper(model, i, j):
        return model.t[j] <= (1 - model.x[i, j]) * (big_m + 1) + model.s[i, j]
    
    def cutoff_lower(model, i, j):
        rank_ij = model.r[i, j]
        prefix = sum(model.x[i, h] for (ii, h) in model.E if ii == i and model.r[ii, h] <= rank_ij)
        return model.s[i, j] + epsilon <= model.t[j] + prefix * (big_m + 1)

    def reject_indicator(model, j):
        return model.u[j] * model.f[j] <= sum(model.x[i, j2] for (i, j2) in model.E if j2 == j)
    
    def cutoff_zero_if_no_reject(model, j):
        return model.t[j] <= model.f[j] * (big_m + 1)

    model.CutoffUpper = pyo.Constraint(model.E, rule=cutoff_upper)
    model.CutoffLower = pyo.Constraint(model.E, rule=cutoff_lower)
    model.RejectIndicator = pyo.Constraint(model.C, rule=reject_indicator)
    model.CutoffZeroIfNoReject = pyo.Constraint(model.C, rule=cutoff_zero_if_no_reject)

    # --- Objective
    def objective(model):
        return sum(model.r[i, j] * model.x[i, j] for (i, j) in model.E)

    model.Objective = pyo.Objective(rule=objective, sense=pyo.minimize)
    return model

In [156]:
so_nw_cut_small = build_model(data_strict_small, formulation="SO-NW-CUT")
_ = solver.solve(so_nw_cut_small)
print_model_summary(so_nw_cut_small)


MODEL SUMMARY
Objective value: 7.00
Students assigned: 8 out of 10
Average rank: 0.88
Total rank sum: 7

College loads: {0: 2, 1: 2, 2: 1, 3: 2, 4: 1}
Cutoff scores: {0: 166, 1: 132, 2: 136, 3: 161, 4: 138}

Assignments:
  Student 0 to College 1 (rank 0)
  Student 1 to College 0 (rank 0)
  Student 4 to College 2 (rank 1)
  Student 5 to College 1 (rank 0)
  Student 6 to College 4 (rank 1)
  Student 7 to College 3 (rank 0)
  Student 8 to College 3 (rank 2)
  Student 9 to College 0 (rank 3)
Unassigned students: [2, 3]



In [157]:
so_nw_cut_medium = build_model(data_strict_small, formulation="SO-NW-CUT")
_ = solver.solve(so_nw_cut_medium)
print_model_summary(so_nw_cut_medium, print_assignments=False)


MODEL SUMMARY
Objective value: 7.00
Students assigned: 8 out of 10
Average rank: 0.88
Total rank sum: 7

College loads: {0: 2, 1: 2, 2: 1, 3: 2, 4: 1}
Cutoff scores: {0: 166, 1: 132, 2: 136, 3: 161, 4: 138}
Unassigned students: [2, 3]



### MIN-CUT Formulation

In [158]:
def _build_min_cut(model: pyo.ConcreteModel, data: Dict[str, Any]) -> pyo.ConcreteModel:
    # --- Cutoff variables and parameters
    scores = data["scores"]
    big_m = max(scores.values()) + 2
    epsilon = 1e-6

    model.t = pyo.Var(model.C, within=pyo.NonNegativeReals, bounds=(0, big_m))

    # --- Cutoff constraints (5) and (6)
    def cutoff_upper(model, i, j):
        return model.t[j] <= (1 - model.x[i, j]) * (big_m + 1) + model.s[i, j]

    def cutoff_lower(model, i, j):
        rank_ij = model.r[i, j]
        prefix = sum(model.x[i, h] for (ii, h) in model.E if ii == i and model.r[ii, h] <= rank_ij)
        return model.s[i, j] + epsilon <= model.t[j] + prefix * (big_m + 1)

    model.CutoffUpper = pyo.Constraint(model.E, rule=cutoff_upper)
    model.CutoffLower = pyo.Constraint(model.E, rule=cutoff_lower)

    # --- Objective
    def objective(model):
        return sum(model.t[j] for j in model.C)

    model.Objective = pyo.Objective(rule=objective, sense=pyo.minimize)
    return model

In [159]:
min_cut_small = build_model(data_strict_small, formulation="MIN-CUT")
_ = solver.solve(min_cut_small)
print_model_summary(min_cut_small)


MODEL SUMMARY
Objective value: 541.00
Students assigned: 8 out of 10
Average rank: 0.88
Total rank sum: 7

College loads: {0: 2, 1: 2, 2: 1, 3: 2, 4: 1}
Cutoff scores: {0: 166, 1: 132, 2: 136, 3: 161, 4: 138}

Assignments:
  Student 0 to College 1 (rank 0)
  Student 1 to College 0 (rank 0)
  Student 4 to College 2 (rank 1)
  Student 5 to College 1 (rank 0)
  Student 6 to College 4 (rank 1)
  Student 7 to College 3 (rank 0)
  Student 8 to College 3 (rank 2)
  Student 9 to College 0 (rank 3)
Unassigned students: [2, 3]



In [160]:
min_cut_medium = build_model(data_strict_medium, formulation="MIN-CUT")
_ = solver.solve(min_cut_medium)
print_model_summary(min_cut_medium, print_assignments=False)


MODEL SUMMARY
Objective value: 16739.00
Students assigned: 50 out of 50
Average rank: 0.70
Total rank sum: 35

College loads: {0: 2, 1: 2, 2: 4, 3: 1, 4: 2, 5: 4, 6: 4, 7: 3, 8: 2, 9: 1, 10: 4, 11: 2, 12: 3, 13: 2, 14: 3, 15: 1, 16: 3, 17: 4, 18: 1, 19: 2}
Cutoff scores: {0: 923, 1: 1805, 2: 1185, 3: 416, 4: 1959, 5: 934, 6: 279, 7: 1829, 8: 2106, 9: 2397, 10: 1633, 11: 2531, 12: 699, 13: 2305, 14: 1108, 15: 85, 16: 712, 17: 319, 18: 2479, 19: 1966}



### MSMR-CUT Formulation

In [161]:
def _build_msmr_cut(model: pyo.ConcreteModel, data: Dict[str, Any]) -> pyo.ConcreteModel:
    # --- Cutoff variables and parameters
    scores = data["scores"]
    big_m = max(scores.values()) + 2
    epsilon = 1e-6
    max_rank = max(model.r[i, j] for (i, j) in model.E)
    K = max_rank + 1

    model.t = pyo.Var(model.C, within=pyo.NonNegativeReals, bounds=(0, big_m))

    # --- Cutoff constraints (5)
    def cutoff_upper(model, i, j):
        return model.t[j] <= (1 - model.x[i, j]) * (big_m + 1) + model.s[i, j]

    model.CutoffUpper = pyo.Constraint(model.E, rule=cutoff_upper)

    # -- Cutoff lower constraints (6)
    def cutoff_lower(model, i, j):
        rank_ij = model.r[i, j]
        prefix = sum(model.x[i, h] for (ii, h) in model.E if ii == i and model.r[ii, h] <= rank_ij)
        return model.s[i, j] + epsilon <= model.t[j] + prefix * (big_m + 1)

    model.CutoffLower = pyo.Constraint(model.E, rule=cutoff_lower)

    # --- Objective
    def objective(model):
        return sum((K - model.r[i, j]) * model.x[i, j] for (i, j) in model.E)

    model.Objective = pyo.Objective(rule=objective, sense=pyo.maximize)
    return model

In [162]:
msmr_cut_small = build_model(data_strict_small, formulation="MSMR-CUT")
_ = solver.solve(msmr_cut_small)
print_model_summary(msmr_cut_small)


MODEL SUMMARY
Objective value: 33.00
Students assigned: 8 out of 10
Average rank: 0.88
Total rank sum: 7

College loads: {0: 2, 1: 2, 2: 1, 3: 2, 4: 1}
Cutoff scores: {0: 166, 1: 132, 2: 136, 3: 161, 4: 138}

Assignments:
  Student 0 to College 1 (rank 0)
  Student 1 to College 0 (rank 0)
  Student 4 to College 2 (rank 1)
  Student 5 to College 1 (rank 0)
  Student 6 to College 4 (rank 1)
  Student 7 to College 3 (rank 0)
  Student 8 to College 3 (rank 2)
  Student 9 to College 0 (rank 3)
Unassigned students: [2, 3]



In [163]:
msmr_cut_medium = build_model(data_strict_medium, formulation="MSMR-CUT")
_ = solver.solve(msmr_cut_medium)
print_model_summary(msmr_cut_medium, print_assignments=False)


MODEL SUMMARY
Objective value: 965.00
Students assigned: 50 out of 50
Average rank: 0.70
Total rank sum: 35

College loads: {0: 2, 1: 2, 2: 4, 3: 1, 4: 2, 5: 4, 6: 4, 7: 3, 8: 2, 9: 1, 10: 4, 11: 2, 12: 3, 13: 2, 14: 3, 15: 1, 16: 3, 17: 4, 18: 1, 19: 2}
Cutoff scores: {0: 923, 1: 1805, 2: 1185, 3: 416, 4: 1959, 5: 934, 6: 279, 7: 1829, 8: 2106, 9: 2397, 10: 1633, 11: 2531, 12: 699, 13: 2305, 14: 1108, 15: 85, 16: 712, 17: 319, 18: 2479, 19: 1966}



### SO-NW-BIN-CUT Formulation

In [164]:
def _build_so_nw_bin_cut(model: pyo.ConcreteModel, data: Dict[str, Any]) -> pyo.ConcreteModel:
    # --- Cutoff variables and parameters
    scores_by_college = _score_lists(data)
    score_pairs = [(j, score) for j in range(data["m"]) for score in scores_by_college[j]]
    model.TS = pyo.Set(initialize=score_pairs, dimen=2)
    model.t = pyo.Var(model.TS, within=pyo.Binary)

    # --- Cutoff constraints (5) and (6)
    def cutoff_ge_accept(model, i, j):
        sc = model.s[i, j]
        return model.x[i, j] <= model.t[j, sc]
    
    model.CutoffGeAccept = pyo.Constraint(model.E, rule=cutoff_ge_accept)

    # --- Monotonicity constraints (7)
    def monotonicity(model, j):
        score_list = scores_by_college[j]
        exprs = []
        for k in range(len(score_list) - 1):
            exprs.append(model.t[j, score_list[k]] <= model.t[j, score_list[k + 1]])
        return exprs

    monotonicity_counter = 0
    for j in range(data["m"]):
        for expr in monotonicity(model, j):
            model.add_component(f"Monotonicity_{j}_{monotonicity_counter}", pyo.Constraint(expr=expr))
            monotonicity_counter += 1

    # --- Envy constraints (8)
    def envy_rule(model, i, j):
        rank_ij = model.r[i, j]
        sum_x = sum(model.x[i, h] for (ii, h) in model.E if ii == i and model.r[ii, h] <= rank_ij)
        sc = model.s[i, j]
        return 1 <= sum_x + (1 - model.t[j, sc])
    
    model.Envy = pyo.Constraint(model.E, rule=envy_rule)

    # --- Lower bound constraints (9)
    def lower_bound_rule(model, j):
        score_list = scores_by_college[j]
        if not score_list:
            return pyo.Constraint.Skip
        return (1 - model.t[j, score_list[0]]) * model.u[j] <= sum(model.x[i, j2] for (i, j2) in model.E if j2 == j)

    model.LowerBound = pyo.Constraint(model.C, rule=lower_bound_rule)

    # --- Objective
    def objective(model):
        return sum(model.r[i, j] * model.x[i, j] for (i, j) in model.E)

    model.Objective = pyo.Objective(rule=objective, sense=pyo.minimize)
    return model

In [165]:
so_nw_bin_cut_small = build_model(data_strict_small, formulation="SO-NW-BIN-CUT")
_ = solver.solve(so_nw_bin_cut_small)
print_model_summary(so_nw_bin_cut_small)


MODEL SUMMARY
Objective value: 7.00
Students assigned: 8 out of 10
Average rank: 0.88
Total rank sum: 7

College loads: {0: 2, 1: 2, 2: 1, 3: 2, 4: 1}
Cutoff scores: {0: 166, 1: 132, 2: 136, 3: 161, 4: 138}

Assignments:
  Student 0 to College 1 (rank 0)
  Student 1 to College 0 (rank 0)
  Student 4 to College 2 (rank 1)
  Student 5 to College 1 (rank 0)
  Student 6 to College 4 (rank 1)
  Student 7 to College 3 (rank 0)
  Student 8 to College 3 (rank 2)
  Student 9 to College 0 (rank 3)
Unassigned students: [2, 3]



In [166]:
so_nw_bin_cut_medium = build_model(data_strict_medium, formulation="SO-NW-BIN-CUT")
_ = solver.solve(so_nw_bin_cut_medium)
print_model_summary(so_nw_bin_cut_medium, print_assignments=False)


MODEL SUMMARY
Objective value: 35.00
Students assigned: 50 out of 50
Average rank: 0.70
Total rank sum: 35

College loads: {0: 2, 1: 2, 2: 4, 3: 1, 4: 2, 5: 4, 6: 4, 7: 3, 8: 2, 9: 1, 10: 4, 11: 2, 12: 3, 13: 2, 14: 3, 15: 1, 16: 3, 17: 4, 18: 1, 19: 2}
Cutoff scores: {0: 923, 1: 1805, 2: 1185, 3: 416, 4: 1959, 5: 934, 6: 279, 7: 1829, 8: 2106, 9: 2397, 10: 1633, 11: 2531, 12: 699, 13: 2305, 14: 1108, 15: 85, 16: 712, 17: 319, 18: 2479, 19: 1966}



### MIN-BIN CUT Formulation

In [167]:
def _build_min_bin_cut(model: pyo.ConcreteModel, data: Dict[str, Any]) -> pyo.ConcreteModel:
    # --- Cutoff variables and parameters
    scores_by_college = _score_lists(data)
    score_pairs = [(j, score) for j in range(data["m"]) for score in scores_by_college[j]]
    model.TS = pyo.Set(initialize=score_pairs, dimen=2)
    model.t = pyo.Var(model.TS, within=pyo.Binary)

    # --- Cutoff constraints (5) and (6)
    def cutoff_ge_accept(model, i, j):
        sc = model.s[i, j]
        return model.x[i, j] <= model.t[j, sc]

    model.CutoffGeAccept = pyo.Constraint(model.E, rule=cutoff_ge_accept)

    # --- Monotonicity constraints (7)
    def monotonicity(model, j):
        score_list = scores_by_college[j]
        exprs = []
        for k in range(len(score_list) - 1):
            exprs.append(model.t[j, score_list[k]] <= model.t[j, score_list[k + 1]])
        return exprs

    monotonicity_counter = 0
    for j in range(data["m"]):
        for expr in monotonicity(model, j):
            model.add_component(f"Monotonicity_{j}_{monotonicity_counter}", pyo.Constraint(expr=expr))
            monotonicity_counter += 1

    # --- Envy constraints (8)
    def envy_rule(model, i, j):
        rank_ij = model.r[i, j]
        sum_x = sum(model.x[i, h] for (ii, h) in model.E if ii == i and model.r[ii, h] <= rank_ij)
        sc = model.s[i, j]
        return 1 <= sum_x + (1 - model.t[j, sc])

    model.Envy = pyo.Constraint(model.E, rule=envy_rule)

    # --- Objective
    def objective(model):
        return sum(model.t[j, sc] for (j, sc) in model.TS)

    model.Objective = pyo.Objective(rule=objective, sense=pyo.maximize)
    return model

In [168]:
min_bin_cut_small = build_model(data_strict_small, formulation="MIN-BIN-CUT")
_ = solver.solve(min_bin_cut_small)
print_model_summary(min_bin_cut_small)


MODEL SUMMARY
Objective value: 19.00
Students assigned: 8 out of 10
Average rank: 0.88
Total rank sum: 7

College loads: {0: 2, 1: 2, 2: 1, 3: 2, 4: 1}
Cutoff scores: {0: 166, 1: 132, 2: 136, 3: 161, 4: 138}

Assignments:
  Student 0 to College 1 (rank 0)
  Student 1 to College 0 (rank 0)
  Student 4 to College 2 (rank 1)
  Student 5 to College 1 (rank 0)
  Student 6 to College 4 (rank 1)
  Student 7 to College 3 (rank 0)
  Student 8 to College 3 (rank 2)
  Student 9 to College 0 (rank 3)
Unassigned students: [2, 3]



In [169]:
min_bin_cut_medium = build_model(data_strict_medium, formulation="MIN-BIN-CUT")
_ = solver.solve(min_bin_cut_medium)
print_model_summary(min_bin_cut_medium, print_assignments=False)


MODEL SUMMARY
Objective value: 682.00
Students assigned: 50 out of 50
Average rank: 0.70
Total rank sum: 35

College loads: {0: 2, 1: 2, 2: 4, 3: 1, 4: 2, 5: 4, 6: 4, 7: 3, 8: 2, 9: 1, 10: 4, 11: 2, 12: 3, 13: 2, 14: 3, 15: 1, 16: 3, 17: 4, 18: 1, 19: 2}
Cutoff scores: {0: 923, 1: 1805, 2: 1185, 3: 416, 4: 1959, 5: 934, 6: 279, 7: 1829, 8: 2106, 9: 2397, 10: 1633, 11: 2531, 12: 699, 13: 2305, 14: 1108, 15: 85, 16: 712, 17: 319, 18: 2479, 19: 1966}



### MSMR-BIN-CUT Formulation

In [170]:
def _build_msmr_bin_cut(model: pyo.ConcreteModel, data: Dict[str, Any]) -> pyo.ConcreteModel:
    # --- Cutoff variables and parameters
    max_rank = max(model.r[i, j] for (i, j) in model.E)
    K = max_rank + 1

    scores_by_college = _score_lists(data)
    score_pairs = [(j, score) for j in range(data["m"]) for score in scores_by_college[j]]
    model.TS = pyo.Set(initialize=score_pairs, dimen=2)
    model.t = pyo.Var(model.TS, within=pyo.Binary)

    # --- Cutoff constraints (5) and (6)
    def cutoff_ge_accept(model, i, j):
        sc = model.s[i, j]
        return model.x[i, j] <= model.t[j, sc]

    model.CutoffGeAccept = pyo.Constraint(model.E, rule=cutoff_ge_accept)

    # --- Monotonicity constraints (7)
    def monotonicity(model, j):
        score_list = scores_by_college[j]
        exprs = []
        for k in range(len(score_list) - 1):
            exprs.append(model.t[j, score_list[k]] <= model.t[j, score_list[k + 1]])
        return exprs
    
    monotonicity_counter = 0
    for j in range(data["m"]):
        for expr in monotonicity(model, j):
            model.add_component(f"Monotonicity_{j}_{monotonicity_counter}", pyo.Constraint(expr=expr))
            monotonicity_counter += 1

    # --- Envy constraints (8)
    def envy_rule(model, i, j):
        rank_ij = model.r[i, j]
        sum_x = sum(model.x[i, h] for (ii, h) in model.E if ii == i and model.r[ii, h] <= rank_ij)
        sc = model.s[i, j]
        return 1 <= sum_x + (1 - model.t[j, sc])

    model.Envy = pyo.Constraint(model.E, rule=envy_rule)

    # --- Objective
    def objective(model):
        return sum((K - model.r[i, j]) * model.x[i, j] for (i, j) in model.E)

    model.Objective = pyo.Objective(rule=objective, sense=pyo.maximize)
    return model

In [171]:
msmr_bin_cut_small = build_model(data_strict_small, formulation="MSMR-BIN-CUT")
_ = solver.solve(msmr_bin_cut_small)
print_model_summary(msmr_bin_cut_small)


MODEL SUMMARY
Objective value: 33.00
Students assigned: 8 out of 10
Average rank: 0.88
Total rank sum: 7

College loads: {0: 2, 1: 2, 2: 1, 3: 2, 4: 1}
Cutoff scores: {0: 166, 1: 132, 2: 136, 3: 161, 4: 138}

Assignments:
  Student 0 to College 1 (rank 0)
  Student 1 to College 0 (rank 0)
  Student 4 to College 2 (rank 1)
  Student 5 to College 1 (rank 0)
  Student 6 to College 4 (rank 1)
  Student 7 to College 3 (rank 0)
  Student 8 to College 3 (rank 2)
  Student 9 to College 0 (rank 3)
Unassigned students: [2, 3]



In [172]:
msmr_bin_cut_medium = build_model(data_strict_medium, formulation="MSMR-BIN-CUT")
_ = solver.solve(msmr_bin_cut_medium)
print_model_summary(msmr_bin_cut_medium, print_assignments=False)


MODEL SUMMARY
Objective value: 965.00
Students assigned: 50 out of 50
Average rank: 0.70
Total rank sum: 35

College loads: {0: 2, 1: 2, 2: 4, 3: 1, 4: 2, 5: 4, 6: 4, 7: 3, 8: 2, 9: 1, 10: 4, 11: 2, 12: 3, 13: 2, 14: 3, 15: 1, 16: 3, 17: 4, 18: 1, 19: 2}
Cutoff scores: {0: 923, 1: 1805, 2: 1185, 3: 416, 4: 1959, 5: 934, 6: 279, 7: 1829, 8: 2106, 9: 2397, 10: 1633, 11: 2531, 12: 699, 13: 2305, 14: 1108, 15: 85, 16: 712, 17: 319, 18: 2479, 19: 1966}



### MSMR-EF Formulation

In [173]:
def _build_msmr_ef(model: pyo.ConcreteModel, data: Dict[str, Any]) -> pyo.ConcreteModel:
    max_rank = max(model.r[i, j] for (i, j) in model.E)
    K = max_rank + 1

    # --- Envy constraints
    def envy_free_rule(model, i, j, h):
        if (i, j) not in model.E or (h, j) not in model.E:
            return pyo.Constraint.Skip
        if model.s[i, j] < model.s[h, j]:
            return pyo.Constraint.Skip
        rank_ij = model.r[i, j]
        sum_x = sum(model.x[i, k] for (ii, k) in model.E if ii == i and model.r[ii, k] <= rank_ij)
        return sum_x >= model.x[h, j]

    model.EnvyConstraints = pyo.ConstraintList()
    for (i, j) in model.E:
        for (h, j2) in model.E:
            if j2 == j and model.s[i, j] >= model.s[h, j]:
                rank_ij = model.r[i, j]
                sum_x = sum(model.x[i, k] for (ii, k) in model.E if ii == i and model.r[ii, k] <= rank_ij)
                model.EnvyConstraints.add(sum_x >= model.x[h, j])

    # --- Objective
    def objective(model):
        return sum((K - model.r[i, j]) * model.x[i, j] for (i, j) in model.E)

    model.Objective = pyo.Objective(rule=objective, sense=pyo.maximize)
    return model

In [174]:
msmr_ef_small = build_model(data_strict_small, formulation="MSMR-EF")
_ = solver.solve(msmr_ef_small)
print_model_summary(msmr_ef_small)


MODEL SUMMARY
Objective value: 33.00
Students assigned: 8 out of 10
Average rank: 0.88
Total rank sum: 7

College loads: {0: 2, 1: 2, 2: 1, 3: 2, 4: 1}
Cutoff scores: {0: 166, 1: 132, 2: 136, 3: 161, 4: 138}

Assignments:
  Student 0 to College 1 (rank 0)
  Student 1 to College 0 (rank 0)
  Student 4 to College 2 (rank 1)
  Student 5 to College 1 (rank 0)
  Student 6 to College 4 (rank 1)
  Student 7 to College 3 (rank 0)
  Student 8 to College 3 (rank 2)
  Student 9 to College 0 (rank 3)
Unassigned students: [2, 3]



In [175]:
msmr_ef_medium = build_model(data_strict_medium, formulation="MSMR-EF")
_ = solver.solve(msmr_ef_medium)
print_model_summary(msmr_ef_medium, print_assignments=False)


MODEL SUMMARY
Objective value: 965.00
Students assigned: 50 out of 50
Average rank: 0.70
Total rank sum: 35

College loads: {0: 2, 1: 2, 2: 4, 3: 1, 4: 2, 5: 4, 6: 4, 7: 3, 8: 2, 9: 1, 10: 4, 11: 2, 12: 3, 13: 2, 14: 3, 15: 1, 16: 3, 17: 4, 18: 1, 19: 2}
Cutoff scores: {0: 923, 1: 1805, 2: 1185, 3: 416, 4: 1959, 5: 934, 6: 279, 7: 1829, 8: 2106, 9: 2397, 10: 1633, 11: 2531, 12: 699, 13: 2305, 14: 1108, 15: 85, 16: 712, 17: 319, 18: 2479, 19: 1966}



## Section 3 - Models with Ties

### SO-H-NW-CUT Formulation

In [176]:
def _build_so_h_nw_cut(model: pyo.ConcreteModel, data: Dict[str, Any]) -> pyo.ConcreteModel:
    """
    SO-H-NW-CUT: Student-Optimal Hungarian Non-Wasteful Cutoff (continuous).
    For ties under Hungarian policy.
    Variables: x (binary), t_j (continuous), f_j (binary), d_{ij} (binary).
    Constraints: (1),(2),(5),(6),(8),(17),(18),(19). Objective (10).
    """
    # --- variables and parameters
    scores = data["scores"]
    big_m = max(scores.values()) + 2
    epsilon = 1e-6

    # Cutoff and reject indicator variables
    model.t = pyo.Var(model.C, within=pyo.NonNegativeReals, bounds=(0, big_m))
    model.f = pyo.Var(model.C, within=pyo.Binary)
    model.d = pyo.Var(model.E, within=pyo.Binary)

    # --- Cutoff constraints (5) and (6) ---
    def cutoff_upper(model, i, j):
        return model.t[j] <= (1 - model.x[i, j]) * (big_m + 1) + model.s[i, j]

    def cutoff_lower(model, i, j):
        rank_ij = model.r[i, j]
        prefix = sum(model.x[i, h] for (ii, h) in model.E if ii == i and model.r[ii, h] <= rank_ij)
        return model.s[i, j] + epsilon <= model.t[j] + prefix * (big_m + 1)

    model.CutoffUpper = pyo.Constraint(model.E, rule=cutoff_upper)
    model.CutoffLower = pyo.Constraint(model.E, rule=cutoff_lower)

    # --- Constraint (8)
    def cutoff_zero_if_no_reject(model, j):
        return model.t[j] <= model.f[j] * (big_m + 1)

    model.CutoffZeroIfNoReject = pyo.Constraint(model.C, rule=cutoff_zero_if_no_reject)

    # --- Constraint (17)
    model.d_blocking_constraint = pyo.ConstraintList()
    for i in range(data["n"]):
        pref_i = data["preferences"][i]
        for rank_idx_j, j in enumerate(pref_i):
            for rank_idx_k, k in enumerate(pref_i):
                if rank_idx_k >= rank_idx_j:
                    if (i, j) in model.E and (i, k) in model.E:
                        model.d_blocking_constraint.add(
                            model.d[i, k] <= 1 - model.x[i, j]
                        )

    # --- Constraint (18)
    def d_cutoff_relation(model, i, j):
        return model.t[j] - 1 <= (1 - model.d[i, j]) * (big_m + 1) + model.s[i, j]

    model.DCutoffRelation = pyo.Constraint(model.E, rule=d_cutoff_relation)

    # --- Constraint (19)
    def non_wastefulness(model, j):
        return model.f[j] * (model.u[j] + 1) <= sum(
            model.x[i, j2] + model.d[i, j2]
            for (i, j2) in model.E
            if j2 == j
        )

    model.NonWastefulness = pyo.Constraint(model.C, rule=non_wastefulness)

    max_rank = max(model.r[i, j] for (i, j) in model.E)
    K = max_rank + 1

    # --- Objective (10)
    def objective(model):
        return sum((K - model.r[i, j]) * model.x[i, j] for (i, j) in model.E)

    model.Objective = pyo.Objective(rule=objective, sense=pyo.maximize)

    return model

In [177]:
so_h_nw_cut_small = build_model(data_small, formulation="SO-H-NW-CUT")
_ = solver.solve(so_h_nw_cut_small)
print_model_summary(so_h_nw_cut_small)


MODEL SUMMARY
Objective value: 19.00
Students assigned: 6 out of 10
Average rank: 1.83
Total rank sum: 11

College loads: {0: 1, 1: 0, 2: 2, 3: 2, 4: 1}
Cutoff scores: {0: 18, 1: 0, 2: 14, 3: 17, 4: 19}

Assignments:
  Student 2 to College 3 (rank 1)
  Student 3 to College 4 (rank 2)
  Student 5 to College 3 (rank 2)
  Student 7 to College 2 (rank 4)
  Student 8 to College 2 (rank 2)
  Student 9 to College 0 (rank 0)
Unassigned students: [0, 1, 4, 6]



In [178]:
so_h_nw_cut_medium = build_model(data_medium, formulation="SO-H-NW-CUT")
_ = solver.solve(so_h_nw_cut_medium)
print_model_summary(so_h_nw_cut_medium, print_assignments=False)


MODEL SUMMARY
Objective value: 966.00
Students assigned: 50 out of 50
Average rank: 0.68
Total rank sum: 34

College loads: {0: 3, 1: 2, 2: 4, 3: 4, 4: 3, 5: 2, 6: 2, 7: 2, 8: 2, 9: 3, 10: 4, 11: 4, 12: 3, 13: 2, 14: 2, 15: 3, 16: 1, 17: 1, 18: 1, 19: 2}
Cutoff scores: {0: 26, 1: 31, 2: 1, 3: 6, 4: 6, 5: 27, 6: 7, 7: 30, 8: 35, 9: 31, 10: 1, 11: 8, 12: 40, 13: 9, 14: 34, 15: 9, 16: 48, 17: 7, 18: 29, 19: 33}



### SO-H-NW-BIN-CUT Formulation

In [179]:
def _build_so_h_nw_bin_cut(model: pyo.ConcreteModel, data: Dict[str, Any]) -> pyo.ConcreteModel:
    """
    SO-H-NW-BIN-CUT: Student-Optimal Hungarian Non-Wasteful Binary Cutoff.
    For ties under Hungarian policy.
    Variables: x (binary), t_j^k (binary), d_{ij} (binary).
    Constraints: (1),(2),(11),(12),(13),(17),(20),(21). Objective (10).
    """
    # --- variables and parameters
    scores_by_college = _score_lists(data)
    score_pairs = [(j, score) for j in range(data["m"]) for score in scores_by_college[j]]
    model.TS = pyo.Set(initialize=score_pairs, dimen=2)
    model.t = pyo.Var(model.TS, within=pyo.Binary)

    model.d = pyo.Var(model.E, within=pyo.Binary)

    # --- Cutoff constraints (11)
    def cutoff_ge_accept(model, i, j):
        sc = model.s[i, j]
        return model.x[i, j] <= model.t[j, sc]

    model.CutoffGeAccept = pyo.Constraint(model.E, rule=cutoff_ge_accept)

    # --- Monotonicity constraints (12)
    def monotonicity(model, j):
        score_list = scores_by_college[j]
        exprs = []
        for k in range(len(score_list) - 1):
            exprs.append(model.t[j, score_list[k]] <= model.t[j, score_list[k + 1]])
        return exprs

    monotonicity_counter = 0
    for j in range(data["m"]):
        for expr in monotonicity(model, j):
            model.add_component(f"Monotonicity_{j}_{monotonicity_counter}", pyo.Constraint(expr=expr))
            monotonicity_counter += 1

    # --- Constraint (13)
    def envy_rule(model, i, j):
        rank_ij = model.r[i, j]
        sum_x = sum(model.x[i, h] for (ii, h) in model.E if ii == i and model.r[ii, h] <= rank_ij)
        sc = model.s[i, j]
        return 1 <= sum_x + (1 - model.t[j, sc])

    model.Envy = pyo.Constraint(model.E, rule=envy_rule)

    # --- Constraint (17)
    model.d_blocking_constraint = pyo.ConstraintList()
    for i in range(data["n"]):
        pref_i = data["preferences"][i]
        for rank_idx_j, j in enumerate(pref_i):
            for rank_idx_k, k in enumerate(pref_i):
                if rank_idx_k >= rank_idx_j:
                    if (i, j) in model.E and (i, k) in model.E:
                        model.d_blocking_constraint.add(
                            model.d[i, k] <= 1 - model.x[i, j]
                        )

    # --- Constraint (21)
    model.d_cutoff_relation_bin = pyo.ConstraintList()
    for (i, j) in model.E:
        sc = model.s[i, j]
        score_list = scores_by_college[j]
        if sc in score_list:
            k = score_list.index(sc)
            if k < len(score_list) - 1:
                model.d_cutoff_relation_bin.add(
                    model.d[i, j] <= model.t[j, score_list[k+1]] - model.t[j, sc]
                )

    # --- Constraint (20)
    def non_wastefulness_bin(model, j):
        score_list = scores_by_college[j]
        if not score_list:
            return pyo.Constraint.Skip
        return (1 - model.t[j, score_list[0]]) * (model.u[j] + 1) <= sum(
            model.x[i, j2] + model.d[i, j2]
            for (i, j2) in model.E
            if j2 == j
        )

    model.NonWastefulnessBin = pyo.Constraint(model.C, rule=non_wastefulness_bin)

    max_rank = max(model.r[i, j] for (i, j) in model.E)
    K = max_rank + 1

    # --- Objective (10)
    def objective(model):
        return sum((K - model.r[i, j]) * model.x[i, j] for (i, j) in model.E)

    model.Objective = pyo.Objective(rule=objective, sense=pyo.maximize)

    return model

In [180]:
so_h_nw_bin_cut_small = build_model(data_small, formulation="SO-H-NW-BIN-CUT")
_ = solver.solve(so_h_nw_bin_cut_small)
print_model_summary(so_h_nw_bin_cut_small)


MODEL SUMMARY
Objective value: 19.00
Students assigned: 6 out of 10
Average rank: 1.83
Total rank sum: 11

College loads: {0: 1, 1: 0, 2: 2, 3: 2, 4: 1}
Cutoff scores: {0: 18, 1: 0, 2: 14, 3: 17, 4: 19}

Assignments:
  Student 2 to College 3 (rank 1)
  Student 3 to College 4 (rank 2)
  Student 5 to College 3 (rank 2)
  Student 7 to College 2 (rank 4)
  Student 8 to College 2 (rank 2)
  Student 9 to College 0 (rank 0)
Unassigned students: [0, 1, 4, 6]



In [181]:
so_h_nw_bin_cut_medium = build_model(data_medium, formulation="SO-H-NW-BIN-CUT")
_ = solver.solve(so_h_nw_bin_cut_medium)
print_model_summary(so_h_nw_bin_cut_medium, print_assignments=False)


MODEL SUMMARY
Objective value: 966.00
Students assigned: 50 out of 50
Average rank: 0.68
Total rank sum: 34

College loads: {0: 3, 1: 2, 2: 4, 3: 4, 4: 3, 5: 2, 6: 2, 7: 2, 8: 2, 9: 3, 10: 4, 11: 4, 12: 3, 13: 2, 14: 2, 15: 3, 16: 1, 17: 1, 18: 1, 19: 2}
Cutoff scores: {0: 26, 1: 31, 2: 1, 3: 6, 4: 6, 5: 27, 6: 7, 7: 30, 8: 35, 9: 31, 10: 1, 11: 8, 12: 40, 13: 9, 14: 34, 15: 9, 16: 48, 17: 7, 18: 29, 19: 33}



## Section 4 - Chilean Models

### SO-C-NW-CUT Formulation

In [182]:
def _build_so_c_nw_cut(model: pyo.ConcreteModel, data: Dict[str, Any]) -> pyo.ConcreteModel:
    """
    SO-C-NW-CUT: Student-Optimal Chilean Non-Wasteful Cutoff (continuous).
    Chilean permissive policy: last tied group is all accepted, possibly violating quota.
    Variables: x (binary), t_j (continuous), f_j (binary), dbar_{ij} (binary).
    Constraints: (1),(5),(6),(7),(8),(22),(23),(24). Objective (10) max.
    """
    # --- variables and parameters
    scores = data["scores"]
    big_m = max(scores.values()) + 2
    epsilon = 1e-6

    model.t = pyo.Var(model.C, within=pyo.NonNegativeReals, bounds=(0, big_m))
    model.f = pyo.Var(model.C, within=pyo.Binary)

    model.dbar = pyo.Var(model.E, within=pyo.Binary)

    # --- Constraints (5) and (6)
    def cutoff_upper(model, i, j):
        return model.t[j] <= (1 - model.x[i, j]) * (big_m + 1) + model.s[i, j]

    def cutoff_lower(model, i, j):
        rank_ij = model.r[i, j]
        prefix = sum(model.x[i, h] for (ii, h) in model.E if ii == i and model.r[ii, h] <= rank_ij)
        return model.s[i, j] + epsilon <= model.t[j] + prefix * (big_m + 1)

    model.CutoffUpper = pyo.Constraint(model.E, rule=cutoff_upper)
    model.CutoffLower = pyo.Constraint(model.E, rule=cutoff_lower)

    # --- Constraints (7) and (8)
    def reject_indicator(model, j):
        return model.u[j] * model.f[j] <= sum(model.x[i, j2] for (i, j2) in model.E if j2 == j)

    def cutoff_zero_if_no_reject(model, j):
        return model.t[j] <= model.f[j] * (big_m + 1)

    model.RejectIndicator = pyo.Constraint(model.C, rule=reject_indicator)
    model.CutoffZeroIfNoReject = pyo.Constraint(model.C, rule=cutoff_zero_if_no_reject)

    # --- Constraint (22)
    def dbar_le_x(model, i, j):
        return model.dbar[i, j] <= model.x[i, j]

    model.DbarLeX = pyo.Constraint(model.E, rule=dbar_le_x)

    # --- Constraint (23)
    def dbar_cutoff_relation(model, i, j):
        return (model.dbar[i, j] - 1) * (big_m + 1) + model.s[i, j] <= model.t[j]

    model.DbarCutoffRelation = pyo.Constraint(model.E, rule=dbar_cutoff_relation)

    # --- Constraint (24)
    def non_wastefulness_chile(model, j):
        return sum(model.x[i, j2] - model.dbar[i, j2] for (i, j2) in model.E if j2 == j) <= model.u[j] - 1

    model.NonWastefulnessChile = pyo.Constraint(model.C, rule=non_wastefulness_chile)

    # --- Objective (10)
    max_rank = max(model.r[i, j] for (i, j) in model.E)
    K = max_rank + 1

    def objective(model):
        return sum((K - model.r[i, j]) * model.x[i, j] for (i, j) in model.E)

    model.Objective = pyo.Objective(rule=objective, sense=pyo.maximize)

    # --- Deactivate capacity constraint (2)
    model.Capacity.deactivate()

    return model

In [183]:
so_c_nw_cut_small = build_model(data_small, formulation="SO-C-NW-CUT")
_ = solver.solve(so_c_nw_cut_small)
print_model_summary(so_c_nw_cut_small)


MODEL SUMMARY
Objective value: 28.00
Students assigned: 8 out of 10
Average rank: 1.50
Total rank sum: 12

College loads: {0: 1, 1: 2, 2: 2, 3: 2, 4: 1}
Cutoff scores: {0: 18, 1: 18, 2: 13, 3: 13, 4: 19}

Assignments:
  Student 0 to College 2 (rank 3)
  Student 1 to College 2 (rank 4)
  Student 2 to College 1 (rank 0)
  Student 3 to College 4 (rank 2)
  Student 5 to College 3 (rank 2)
  Student 7 to College 1 (rank 0)
  Student 8 to College 3 (rank 1)
  Student 9 to College 0 (rank 0)
Unassigned students: [4, 6]



In [184]:
so_c_nw_cut_medium = build_model(data_medium, formulation="SO-C-NW-CUT")
_ = solver.solve(so_c_nw_cut_medium)
print_model_summary(so_c_nw_cut_medium, print_assignments=False)


MODEL SUMMARY
Objective value: 966.00
Students assigned: 50 out of 50
Average rank: 0.68
Total rank sum: 34

College loads: {0: 3, 1: 2, 2: 4, 3: 4, 4: 3, 5: 2, 6: 2, 7: 2, 8: 2, 9: 3, 10: 4, 11: 4, 12: 3, 13: 2, 14: 2, 15: 3, 16: 1, 17: 1, 18: 1, 19: 2}
Cutoff scores: {0: 26, 1: 31, 2: 1, 3: 6, 4: 6, 5: 27, 6: 7, 7: 30, 8: 35, 9: 31, 10: 1, 11: 8, 12: 40, 13: 9, 14: 34, 15: 9, 16: 48, 17: 7, 18: 29, 19: 33}



### SO-C-NW-BIN-CUT Formulation

In [185]:
def _build_so_c_nw_bin_cut(model: pyo.ConcreteModel, data: Dict[str, Any]) -> pyo.ConcreteModel:
    """
    SO-C-NW-BIN-CUT: Student-Optimal Chilean Non-Wasteful Binary Cutoff.
    Chilean permissive policy.
    Variables: x (binary), t_j^k (binary), dbar_{ij} (binary).
    Constraints: (1),(11),(12),(13),(22),(24),(25). Objective (10) max.
    """
    scores_by_college = _score_lists(data)
    score_pairs = [(j, score) for j in range(data["m"]) for score in scores_by_college[j]]
    model.TS = pyo.Set(initialize=score_pairs, dimen=2)
    model.t = pyo.Var(model.TS, within=pyo.Binary)

    model.dbar = pyo.Var(model.E, within=pyo.Binary)

    # --- Constraints (11)
    def cutoff_ge_accept(model, i, j):
        sc = model.s[i, j]
        return model.x[i, j] <= model.t[j, sc]

    model.CutoffGeAccept = pyo.Constraint(model.E, rule=cutoff_ge_accept)

    # --- Monotonicity constraints (12)
    def monotonicity(model, j):
        score_list = scores_by_college[j]
        exprs = []
        for k in range(len(score_list) - 1):
            exprs.append(model.t[j, score_list[k]] <= model.t[j, score_list[k + 1]])
        return exprs
    
    monotonicity_counter = 0
    for j in range(data["m"]):
        for expr in monotonicity(model, j):
            model.add_component(f"Monotonicity_{j}_{monotonicity_counter}", pyo.Constraint(expr=expr))
            monotonicity_counter += 1

    # --- Envy constraints (13)
    def envy_rule(model, i, j):
        rank_ij = model.r[i, j]
        sum_x = sum(model.x[i, h] for (ii, h) in model.E if ii == i and model.r[ii, h] <= rank_ij)
        sc = model.s[i, j]
        return 1 <= sum_x + (1 - model.t[j, sc])

    model.Envy = pyo.Constraint(model.E, rule=envy_rule)

    # --- Constraint (22)
    def dbar_le_x(model, i, j):
        return model.dbar[i, j] <= model.x[i, j]

    model.DbarLeX = pyo.Constraint(model.E, rule=dbar_le_x)

    # --- Constraint (24)
    def non_wastefulness_chile_bin(model, j):
        return sum(model.x[i, j2] - model.dbar[i, j2] for (i, j2) in model.E if j2 == j) <= model.u[j] - 1

    model.NonWastefulnessChileBin = pyo.Constraint(model.C, rule=non_wastefulness_chile_bin)

    # --- Constraint (25)
    model.dbar_cutoff_bin = pyo.ConstraintList()
    for (i, j) in model.E:
        sc = model.s[i, j]
        score_list = scores_by_college[j]
        if sc in score_list:
            k = score_list.index(sc)
            if k == 0:
                model.dbar_cutoff_bin.add(model.dbar[i, j] <= model.t[j, sc])
            else:
                prev_score = score_list[k-1]
                model.dbar_cutoff_bin.add(model.dbar[i, j] <= model.t[j, sc] - model.t[j, prev_score])

    # --- Objective (10)
    max_rank = max(model.r[i, j] for (i, j) in model.E)
    K = max_rank + 1

    def objective(model):
        return sum((K - model.r[i, j]) * model.x[i, j] for (i, j) in model.E)

    model.Objective = pyo.Objective(rule=objective, sense=pyo.maximize)

    # --- Deactivate capacity constraint (2)
    model.Capacity.deactivate()

    return model

In [186]:
so_c_nw_bin_cut_small = build_model(data_small, formulation="SO-C-NW-BIN-CUT")
_ = solver.solve(so_c_nw_bin_cut_small)
print_model_summary(so_c_nw_bin_cut_small)


MODEL SUMMARY
Objective value: 28.00
Students assigned: 8 out of 10
Average rank: 1.50
Total rank sum: 12

College loads: {0: 1, 1: 2, 2: 2, 3: 2, 4: 1}
Cutoff scores: {0: 18, 1: 18, 2: 13, 3: 13, 4: 19}

Assignments:
  Student 0 to College 2 (rank 3)
  Student 1 to College 2 (rank 4)
  Student 2 to College 1 (rank 0)
  Student 3 to College 4 (rank 2)
  Student 5 to College 3 (rank 2)
  Student 7 to College 1 (rank 0)
  Student 8 to College 3 (rank 1)
  Student 9 to College 0 (rank 0)
Unassigned students: [4, 6]



In [187]:
so_c_nw_bin_cut_medium = build_model(data_medium, formulation="SO-C-NW-BIN-CUT")
_ = solver.solve(so_c_nw_bin_cut_medium)
print_model_summary(so_c_nw_bin_cut_medium, print_assignments=False)


MODEL SUMMARY
Objective value: 966.00
Students assigned: 50 out of 50
Average rank: 0.68
Total rank sum: 34

College loads: {0: 3, 1: 2, 2: 4, 3: 4, 4: 3, 5: 2, 6: 2, 7: 2, 8: 2, 9: 3, 10: 4, 11: 4, 12: 3, 13: 2, 14: 2, 15: 3, 16: 1, 17: 1, 18: 1, 19: 2}
Cutoff scores: {0: 26, 1: 31, 2: 1, 3: 6, 4: 6, 5: 27, 6: 7, 7: 30, 8: 35, 9: 31, 10: 1, 11: 8, 12: 40, 13: 9, 14: 34, 15: 9, 16: 48, 17: 7, 18: 29, 19: 33}



# Phase 6 - Model Comparison

### Helper Functions

In [188]:
def solve_model(
        data: Dict[str, Any] | str | Path,
        formulation: str = "SO-BB",
        solver_name: str = "cplex",
        tee: bool = False):
    """
    solve a single formulation and return the model and a compact solution summary.
    """
    instance = _normalize_data(data)
    model = build_model(instance, formulation=formulation)
    solver = pyo.SolverFactory(solver_name)
    result = solver.solve(model, tee=tee)
    status = str(result.solver.termination_condition).lower()
    return {
        "model": model,
        "instance": instance,
        "formulation": formulation,
        "status": status,
        "result": result,
    }


def solve_all_models(
        data: Dict[str, Any] | str | Path,
        solver_name: str = "cplex",
        tee: bool = False,
        formulations: List[str] | None = None
        ) -> Dict[str, Dict[str, Any]]:
    """
    solve all the models and return the results for all of them as a dictionary
    """
    instance = _normalize_data(data)
    results = {}
    for formulation in formulations:
        solved = solve_model(instance, formulation=formulation, solver_name=solver_name, tee=tee)
        results[formulation] = solved
    return results

In [189]:
def extract_assignment(
        model: pyo.ConcreteModel,
        data: Dict[str, Any] | None = None
        )-> Dict[int, int | None]:
    if data is None:
        data = {"preferences": {i: [] for i in range(len(model.A))}}
    preferences = data.get("preferences", {})
    assignment = {}
    for i in range(len(preferences)):
        assigned = None
        for j in preferences.get(i, []):
            try:
                if pyo.value(model.x[i, j]) > 0.5:
                    assigned = j
                    break
            except ValueError:
                break
        assignment[i] = assigned
    return assignment

def summarize_results(
        results: Dict[str, Dict[str, Any]]
        ) -> List[Dict[str, Any]]:
    rows = []
    for formulation, info in results.items():
        model = info["model"]
        instance = info["instance"]
        assignment = extract_assignment(model, instance)
        admitted = [j for j in assignment.values() if j is not None]
        college_loads = {j: sum(1 for assigned in assignment.values() if assigned == j) for j in range(instance["m"])}
        try:
            rank_objective = sum(instance["preferences"][i].index(assignment[i]) for i in range(instance["n"]) if assignment[i] is not None)
            model_objective = pyo.value(model.Objective)
        except (ValueError, TypeError):
            rank_objective = None
            model_objective = None
        rows.append(
            {
                "model": formulation,
                "status": info["status"],
                "model_objective": model_objective,
                "rank_objective": rank_objective,
                "students_assigned": len(admitted),
                "college_loads": college_loads,
                "assignments": assignment,
            }
        )
    return rows

def print_summary(
        rows: Iterable[Dict[str, Any]]
        ) -> None:
    for row in rows:
        print(f"{row['model']}: status={row['status']}, model_objective={row['model_objective']}, rank_objective={row['rank_objective']}, assigned={row['students_assigned']}")
        print("  college loads:", row["college_loads"])
        print("  assignments:", row["assignments"])
        print()

def solve_and_summarize_all(
        data: Dict[str, Any] | str | Path,
        solver_name: str = "cplex",
        tee: bool = False,
        formulations: List[str] | None = None
        ) -> List[Dict[str, Any]]:
    """
    Convenience wrapper to solve all formulations and return comparable summaries.
    """
    return summarize_results(solve_all_models(data, solver_name=solver_name, tee=tee, formulations=formulations))

def compute_metrics(assignment, data):
    """Compute fairness and satisfaction metrics from an assignment."""
    preferences = data['preferences']
    n = data['n']
    m = data['m']
    
    matched = [i for i in range(n) if assignment[i] is not None]
    unmatched = [i for i in range(n) if assignment[i] is None]
    
    college_loads = {}
    for j in range(m):
        college_loads[j] = sum(1 for assigned in assignment.values() if assigned == j)
    
    ranks_achieved = []
    for i in matched:
        j = assignment[i]
        rank = preferences[i].index(j)
        ranks_achieved.append(rank)
    
    avg_rank = sum(ranks_achieved) / len(ranks_achieved) if ranks_achieved else None
    max_rank = max(ranks_achieved) if ranks_achieved else None
    
    return {
        'num_matched': len(matched),
        'num_unmatched': len(unmatched),
        'college_loads': college_loads,
        'avg_preference_rank': avg_rank,
        'max_preference_rank': max_rank,
        'total_rank_objective': sum(ranks_achieved)
    }

In [190]:
def get_model_stats(model):
    """
    Return a dict with:
        num_vars: total number of variables
        num_constraints: total number of constraints (including block constraints)
        size_kb: size of the LP file in KB
    """
    # variables
    num_vars = 0
    for var in model.component_objects(pyo.Var, active=True):
        num_vars += len(var)

    # constraints
    num_constraints = 0
    for con in model.component_objects(pyo.Constraint, active=True):
        if con.is_indexed():
            num_constraints += len(con)
        else:
            num_constraints += 1

    # write LP file to temp and get its size
    with tempfile.NamedTemporaryFile(suffix='.lp', delete=False) as f:
        temp_filename = f.name
    try:
        model.write(temp_filename, format='lp')
        size_kb = os.path.getsize(temp_filename) / 1024.0
    finally:
        os.remove(temp_filename)

    return {
        'num_vars': num_vars,
        'num_constraints': num_constraints,
        'size_kb': size_kb,
    }

def break_ties_randomly(data):
    import copy, random
    new_data = copy.deepcopy(data)
    new_scores = {}
    for (i,j), score in data['scores'].items():
        new_scores[(i,j)] = score + random.random() * 1e-5
    new_data['scores'] = new_scores
    return new_data

def compute_policy_metrics(model, data):
    """
    Compute metrics for a given model and data, similar to paper's table 4.
    Returns a dict with:
        size:        number of students assigned
        avg_rank:    average rank of assigned students
        avg_cutoffs: average cutoff scores for colleges (if applicable)
        rejections:  number of applications rejected (student not matched)
    """
    # 1. Size & Ranks
    size = 0
    rank_sum = 0
    for (i, j) in model.E:
        if pyo.value(model.x[i, j]) > 0.5:
            size += 1
            rank_sum += pyo.value(model.r[i, j])

    avg_rank = rank_sum / size if size > 0 else 0

    # 2. Rejections: total students - matched students
    rejections = data['n'] - size

    # 3. Average Cutoffs
    cutoffs = []
    for j in range(data['m']):
        assigned_scores = []
        for (i, col) in model.E:
            if col == j and pyo.value(model.x[i, j]) > 0.5:
                assigned_scores.append(data['scores'][(i, j)])
        if assigned_scores:
            cutoff = min(assigned_scores)
        else:
            cutoff = 0
        cutoffs.append(cutoff)

    avg_cutoffs = sum(cutoffs) / len(cutoffs) if cutoffs else 0

    return {
        'size': size,
        'avg_rank': avg_rank,
        'avg_cutoffs': avg_cutoffs,
        'rejections': rejections
    }

## Section 2

### With Ties

In [191]:
section_two_formulations = ["SO-BB", "SO-NW-CUT", "MIN-CUT", "MSMR-CUT", "SO-NW-BIN-CUT", "MIN-BIN-CUT", "MSMR-BIN-CUT", "MSMR-EF"]

results_small = solve_and_summarize_all(data_small, solver_name='cplex', tee=False, formulations=section_two_formulations)


print("\nSUMMARY TABLE")
print("="*100)

df_summary = pd.DataFrame([
    {
        'Formulation': r['model'],
        'Status': r['status'].upper(),
        'Model Obj': f"{r['model_objective']:.2f}" if r['model_objective'] is not None else "N/A",
        'Rank Obj': f"{r['rank_objective']}" if r['rank_objective'] is not None else "N/A",
        'Matched': r['students_assigned'],
        'Avg Rank': f"{r['rank_objective']/max(1, r['students_assigned']):.2f}" if r['rank_objective'] is not None and r['students_assigned'] > 0 else "N/A"
    }
    for r in results_small
])

print(df_summary.to_string(index=False))

ERROR: evaluating object as numeric value: x[0,0]
        (object: <class 'pyomo.core.base.var.VarData'>)
    No value for uninitialized VarData object x[0,0]
ERROR: evaluating object as numeric value: x[1,3]
        (object: <class 'pyomo.core.base.var.VarData'>)
    No value for uninitialized VarData object x[1,3]
ERROR: evaluating object as numeric value: x[2,1]
        (object: <class 'pyomo.core.base.var.VarData'>)
    No value for uninitialized VarData object x[2,1]
ERROR: evaluating object as numeric value: x[3,0]
        (object: <class 'pyomo.core.base.var.VarData'>)
    No value for uninitialized VarData object x[3,0]
ERROR: evaluating object as numeric value: x[4,2]
        (object: <class 'pyomo.core.base.var.VarData'>)
    No value for uninitialized VarData object x[4,2]
ERROR: evaluating object as numeric value: x[5,1]
        (object: <class 'pyomo.core.base.var.VarData'>)
    No value for uninitialized VarData object x[5,1]
ERROR: evaluating object as numeric value: x[6

### Without Ties

In [192]:
results_small = solve_and_summarize_all(data_strict_small, solver_name='cplex', tee=False, formulations=section_two_formulations)

print("\nSUMMARY TABLE")
print("="*100)

df_summary = pd.DataFrame([
    {
        'Formulation': r['model'],
        'Status': r['status'].upper(),
        'Model Obj': f"{r['model_objective']:.2f}" if r['model_objective'] is not None else "N/A",
        'Rank Obj': f"{r['rank_objective']}" if r['rank_objective'] is not None else "N/A",
        'Matched': r['students_assigned'],
        'Avg Rank': f"{r['rank_objective']/max(1, r['students_assigned']):.2f}" if r['rank_objective'] is not None and r['students_assigned'] > 0 else "N/A"
    }
    for r in results_small
])

print(df_summary.to_string(index=False))


SUMMARY TABLE
  Formulation  Status Model Obj Rank Obj  Matched Avg Rank
        SO-BB OPTIMAL      7.00        7        8     0.88
    SO-NW-CUT OPTIMAL      7.00        7        8     0.88
      MIN-CUT OPTIMAL    541.00        7        8     0.88
     MSMR-CUT OPTIMAL     33.00        7        8     0.88
SO-NW-BIN-CUT OPTIMAL      7.00        7        8     0.88
  MIN-BIN-CUT OPTIMAL     19.00        7        8     0.88
 MSMR-BIN-CUT OPTIMAL     33.00        7        8     0.88
      MSMR-EF OPTIMAL     33.00        7        8     0.88


### Computationsl Comparison (Table 2)

In [193]:
rows = []

for formulation in section_two_formulations:
    print(f"Solving {formulation} on strict medium...")
    start = time.perf_counter()
    info = solve_model(data_strict_medium, formulation=formulation, solver_name='cplex', tee=False)
    elapsed = time.perf_counter() - start
    model = info['model']
    stats = get_model_stats(model)
    rows.append({
        'Formulation': formulation,
        '#variables': stats['num_vars'],
        '#constraints': stats['num_constraints'],
        'size(Kb)': f"{stats['size_kb']:.2f}",
        'run time(s)': f"{elapsed:.2f}"
    })

# create data frame and print
df_table = pd.DataFrame(rows)
print("\nTable 2: The performance of (mixed) integer programming formulations for the classical Gale-Shapley model.")
print("="*100)
print(df_table.to_string(index=False))

Solving SO-BB on strict medium...
not match specified file format (lp)
Solving SO-NW-CUT on strict medium...
not match specified file format (lp)
Solving MIN-CUT on strict medium...
not match specified file format (lp)
Solving MSMR-CUT on strict medium...
not match specified file format (lp)
Solving SO-NW-BIN-CUT on strict medium...
not match specified file format (lp)
Solving MIN-BIN-CUT on strict medium...
not match specified file format (lp)
Solving MSMR-BIN-CUT on strict medium...
not match specified file format (lp)
Solving MSMR-EF on strict medium...
not match specified file format (lp)

Table 2: The performance of (mixed) integer programming formulations for the classical Gale-Shapley model.
  Formulation  #variables  #constraints size(Kb) run time(s)
        SO-BB        1000          1070   337.51        5.03
    SO-NW-CUT        1040          2110   246.70        1.27
      MIN-CUT        1020          2070   225.53        2.10
     MSMR-CUT        1020          2070   236.94

## Section 3

In [194]:
section_three_formulations = ["MIN-CUT", "MSMR-CUT", "MIN-BIN-CUT", "MSMR-BIN-CUT", "SO-H-NW-CUT", "SO-H-NW-BIN-CUT"]

results_small = solve_and_summarize_all(data_small, solver_name='cplex', tee=False, formulations=section_three_formulations)


print("\nSUMMARY TABLE")
print("="*100)

df_summary = pd.DataFrame([
    {
        'Formulation': r['model'],
        'Status': r['status'].upper(),
        'Model Obj': f"{r['model_objective']:.2f}" if r['model_objective'] is not None else "N/A",
        'Rank Obj': f"{r['rank_objective']}" if r['rank_objective'] is not None else "N/A",
        'Matched': r['students_assigned'],
        'Avg Rank': f"{r['rank_objective']/max(1, r['students_assigned']):.2f}" if r['rank_objective'] is not None and r['students_assigned'] > 0 else "N/A"
    }
    for r in results_small
])

print(df_summary.to_string(index=False))


SUMMARY TABLE
    Formulation  Status Model Obj Rank Obj  Matched Avg Rank
        MIN-CUT OPTIMAL     64.00       12        7     1.71
       MSMR-CUT OPTIMAL     23.00        7        6     1.17
    MIN-BIN-CUT OPTIMAL      9.00       11        6     1.83
   MSMR-BIN-CUT OPTIMAL     19.00       11        6     1.83
    SO-H-NW-CUT OPTIMAL     19.00       11        6     1.83
SO-H-NW-BIN-CUT OPTIMAL     19.00       11        6     1.83


In [199]:
rows = []

for formulation in section_three_formulations:
    print(f"Solving {formulation} on strict medium...")
    start = time.perf_counter()
    info = solve_model(data_medium, formulation=formulation, solver_name='cplex', tee=False)
    elapsed = time.perf_counter() - start
    model = info['model']
    stats = get_model_stats(model)
    rows.append({
        'Formulation': formulation,
        '#variables': stats['num_vars'],
        '#constraints': stats['num_constraints'],
        'size(Kb)': f"{stats['size_kb']:.2f}",
        'run time(s)': f"{elapsed:.2f}"
    })

# create data frame and print
df_table = pd.DataFrame(rows)
print("\nTable 3: The performances of (mixed) integer programming formulations for the case of ties.")
print("="*100)
print(df_table.to_string(index=False))

Solving MIN-CUT on strict medium...
not match specified file format (lp)
Solving MSMR-CUT on strict medium...
not match specified file format (lp)
Solving MIN-BIN-CUT on strict medium...
not match specified file format (lp)
Solving MSMR-BIN-CUT on strict medium...
not match specified file format (lp)
Solving SO-H-NW-CUT on strict medium...
not match specified file format (lp)
Solving SO-H-NW-BIN-CUT on strict medium...
not match specified file format (lp)

Table 3: The performances of (mixed) integer programming formulations for the case of ties.
    Formulation  #variables  #constraints size(Kb) run time(s)
        MIN-CUT        1020          2070   199.34        0.69
       MSMR-CUT        1020          2070   210.76        0.87
    MIN-BIN-CUT        1647          2697   232.83        0.70
   MSMR-BIN-CUT        1647          2697   229.61        0.81
    SO-H-NW-CUT        2040         13610   661.14        2.52
SO-H-NW-BIN-CUT        2647         14188   686.69        2.00


## Section 4

In [196]:
section_four_chilean_formulations = ["SO-C-NW-CUT", "SO-C-NW-BIN-CUT"]

results_small = solve_and_summarize_all(data_small, solver_name='cplex', tee=False, formulations=section_four_chilean_formulations)


print("\nSUMMARY TABLE")
print("="*100)

df_summary = pd.DataFrame([
    {
        'Formulation': r['model'],
        'Status': r['status'].upper(),
        'Model Obj': f"{r['model_objective']:.2f}" if r['model_objective'] is not None else "N/A",
        'Rank Obj': f"{r['rank_objective']}" if r['rank_objective'] is not None else "N/A",
        'Matched': r['students_assigned'],
        'Avg Rank': f"{r['rank_objective']/max(1, r['students_assigned']):.2f}" if r['rank_objective'] is not None and r['students_assigned'] > 0 else "N/A"
    }
    for r in results_small
])

print(df_summary.to_string(index=False))


SUMMARY TABLE
    Formulation  Status Model Obj Rank Obj  Matched Avg Rank
    SO-C-NW-CUT OPTIMAL     28.00       12        8     1.50
SO-C-NW-BIN-CUT OPTIMAL     28.00       12        8     1.50


In [197]:
def solve_with_opt(data, formulation, opt_type, solver='cplex'):
    """
    opt_type: 'A-opt' or 'C-opt'
    """
    import pyomo.environ as pyo
    model = build_model(data, formulation=formulation)
    
    if formulation in ["SO-H-NW-BIN-CUT", "SO-C-NW-BIN-CUT"]:
        if opt_type == 'A-opt':
            model.Objective.sense = pyo.maximize
        else:
            model.Objective.sense = pyo.minimize
    elif formulation == "SO-NW-BIN-CUT":
        if opt_type == 'A-opt':
            model.Objective.sense = pyo.minimize
        else:
            model.Objective.sense = pyo.maximize
            
    solver = pyo.SolverFactory(solver)
    result = solver.solve(model, tee=False)
    return model

In [198]:
policies = {
    'Hungarian': 'SO-H-NW-BIN-CUT',
    'Chilean': 'SO-C-NW-BIN-CUT',
    'Irish': 'SO-NW-BIN-CUT'
}

rows = []

data_irish = break_ties_randomly(data_medium)

for policy_name, formulation in policies.items():
    for opt in ['A-opt', 'C-opt']:
        if policy_name == 'Irish':
            # For Irish, use random tie-breaking each time
            data_to_use = data_irish
        else:
            data_to_use = data_medium
        
        model = solve_with_opt(data_to_use, formulation, opt)
        
        metrics = compute_policy_metrics(model, data_to_use)
        rows.append({
            'Policy': policy_name,
            'Opt': opt,
            'Size': metrics['size'],
            'Avg Rank': round(metrics['avg_rank'], 4),
            'Avg Cutoffs': round(metrics['avg_cutoffs'], 4),
            '# Rejections': metrics['rejections']
        })

df = pd.DataFrame(rows)
print("\nTable 4: Comparison of student-optimal (A-opt) and student-pessimal (C-opt) stable matchings")
print(df.to_string(index=False))


Table 4: Comparison of student-optimal (A-opt) and student-pessimal (C-opt) stable matchings
   Policy   Opt  Size  Avg Rank  Avg Cutoffs  # Rejections
Hungarian A-opt    50    0.6800        20.90             0
Hungarian C-opt    49    5.1224        41.55             1
  Chilean A-opt    50    0.6800        20.90             0
  Chilean C-opt     0    0.0000         0.00            50
    Irish A-opt    50    0.6800        20.90             0
    Irish C-opt    50    0.6800        20.90             0


# Phase 7 - Innovations

## Adding Constraint 14 equivalent for Chilean Policy

In [201]:
def _build_so_c_nw_bin_cut(model: pyo.ConcreteModel, data: Dict[str, Any]) -> pyo.ConcreteModel:
    """
    SO-C-NW-BIN-CUT: Student-Optimal Chilean Non-Wasteful Binary Cutoff.
    Chilean permissive policy.
    Variables: x (binary), t_j^k (binary), dbar_{ij} (binary).
    Constraints: (1),(11),(12),(13),(22),(24),(25). Objective (10) max.
    """
    scores_by_college = _score_lists(data)
    score_pairs = [(j, score) for j in range(data["m"]) for score in scores_by_college[j]]
    model.TS = pyo.Set(initialize=score_pairs, dimen=2)
    model.t = pyo.Var(model.TS, within=pyo.Binary)

    model.dbar = pyo.Var(model.E, within=pyo.Binary)

    # --- Constraints (11)
    def cutoff_ge_accept(model, i, j):
        sc = model.s[i, j]
        return model.x[i, j] <= model.t[j, sc]

    model.CutoffGeAccept = pyo.Constraint(model.E, rule=cutoff_ge_accept)

    # --- Monotonicity constraints (12)
    def monotonicity(model, j):
        score_list = scores_by_college[j]
        exprs = []
        for k in range(len(score_list) - 1):
            exprs.append(model.t[j, score_list[k]] <= model.t[j, score_list[k + 1]])
        return exprs
    
    monotonicity_counter = 0
    for j in range(data["m"]):
        for expr in monotonicity(model, j):
            model.add_component(f"Monotonicity_{j}_{monotonicity_counter}", pyo.Constraint(expr=expr))
            monotonicity_counter += 1

    # --- Envy constraints (13)
    def envy_rule(model, i, j):
        rank_ij = model.r[i, j]
        sum_x = sum(model.x[i, h] for (ii, h) in model.E if ii == i and model.r[ii, h] <= rank_ij)
        sc = model.s[i, j]
        return 1 <= sum_x + (1 - model.t[j, sc])

    model.Envy = pyo.Constraint(model.E, rule=envy_rule)

    # --- Constraint (22)
    def dbar_le_x(model, i, j):
        return model.dbar[i, j] <= model.x[i, j]

    model.DbarLeX = pyo.Constraint(model.E, rule=dbar_le_x)

    # --- Constraint (24)
    def non_wastefulness_chile_bin(model, j):
        return sum(model.x[i, j2] - model.dbar[i, j2] for (i, j2) in model.E if j2 == j) <= model.u[j] - 1

    model.NonWastefulnessChileBin = pyo.Constraint(model.C, rule=non_wastefulness_chile_bin)

    # --- Constraint (25)
    model.dbar_cutoff_bin = pyo.ConstraintList()
    for (i, j) in model.E:
        sc = model.s[i, j]
        score_list = scores_by_college[j]
        if sc in score_list:
            k = score_list.index(sc)
            if k == 0:
                model.dbar_cutoff_bin.add(model.dbar[i, j] <= model.t[j, sc])
            else:
                prev_score = score_list[k-1]
                model.dbar_cutoff_bin.add(model.dbar[i, j] <= model.t[j, sc] - model.t[j, prev_score])
    
    # --- Non-wastefulness constraint (14) equivalent for Chilean
    # --- Not in the paper
    def chilean_lower_bound(model, j):
        score_list = scores_by_college[j]
        if not score_list:
            return pyo.Constraint.Skip
        return (1 - model.t[j, score_list[0]]) * model.u[j] <= sum(
            model.x[i, j2] for (i, j2) in model.E if j2 == j
        )
    model.ChileanLowerBound = pyo.Constraint(model.C, rule=chilean_lower_bound)

    # --- Objective (10)
    max_rank = max(model.r[i, j] for (i, j) in model.E)
    K = max_rank + 1

    def objective(model):
        return sum((K - model.r[i, j]) * model.x[i, j] for (i, j) in model.E)

    model.Objective = pyo.Objective(rule=objective, sense=pyo.maximize)

    # --- Deactivate capacity constraint (2)
    model.Capacity.deactivate()

    return model


In [202]:
policies = {
    'Hungarian': 'SO-H-NW-BIN-CUT',
    'Chilean': 'SO-C-NW-BIN-CUT',
    'Irish': 'SO-NW-BIN-CUT'
}

rows = []

data_irish = break_ties_randomly(data_medium)

for policy_name, formulation in policies.items():
    for opt in ['A-opt', 'C-opt']:
        if policy_name == 'Irish':
            # For Irish, use random tie-breaking each time
            data_to_use = data_irish
        else:
            data_to_use = data_medium
        
        model = solve_with_opt(data_to_use, formulation, opt)
        
        metrics = compute_policy_metrics(model, data_to_use)
        rows.append({
            'Policy': policy_name,
            'Opt': opt,
            'Size': metrics['size'],
            'Avg Rank': round(metrics['avg_rank'], 4),
            'Avg Cutoffs': round(metrics['avg_cutoffs'], 4),
            '# Rejections': metrics['rejections']
        })

df = pd.DataFrame(rows)
print("\nTable 4: Comparison of student-optimal (A-opt) and student-pessimal (C-opt) stable matchings")
print(df.to_string(index=False))


Table 4: Comparison of student-optimal (A-opt) and student-pessimal (C-opt) stable matchings
   Policy   Opt  Size  Avg Rank  Avg Cutoffs  # Rejections
Hungarian A-opt    50    0.6800        20.90             0
Hungarian C-opt    49    5.1224        41.55             1
  Chilean A-opt    50    0.6800        20.90             0
  Chilean C-opt    50    0.6800        20.90             0
    Irish A-opt    50    0.6800        20.90             0
    Irish C-opt    50    0.6800        20.90             0
